In [ ]:
# --- Cell 1: load the data and set the service level --------------------------------------
# Goal of this notebook: for every active product, work out how much stock to hold
# (safety stock), when to reorder (reorder point), how well orders were met (fill rate),
# how old the stock is (ageing), and how important and predictable it is (ABC/XYZ).

import os
from pathlib import Path
if Path.cwd().name == "notebooks":                 
    os.chdir("..")                                 

import numpy as np, pandas as pd                   # numpy = maths, pandas = tables
from statistics import NormalDist                  # gives the z-score for a service level

# The daily table built by notebook 01: one row per product per day
df = pd.read_parquet("data/processed/daily_flows.parquet")

# Keep only the 31 active products (the 10 discontinued ones would distort the results)
active = pd.read_parquet("data/processed/product_status.parquet").query("status == 'Active'")["product"]
df = df[df["product"].isin(active)].copy()
df["date"] = pd.to_datetime(df["date"])            # make sure dates are real dates, not text
df = df.sort_values(["product", "date"])           # oldest to newest within each product
last_day = df["date"].max()                        # 9 Aug 2023, the "today" of this analysis

# Service level = how often we want to avoid running out during a replenishment cycle.
# 95% is a common target. The z-score turns it into "how many standard deviations of
# demand to cover": 95% -> 1.645.
SERVICE_LEVEL = 0.95
Z = NormalDist().inv_cdf(SERVICE_LEVEL)
print(f"Service level {SERVICE_LEVEL:.0%} -> z = {Z:.3f}")

In [ ]:
# --- Cell 2: calculate the planning numbers for each product -------------------------------
# We loop over the products one at a time and collect the results in two lists:
#   results       -> one row per product (safety stock, reorder point, fill rate ...)
#   stock_batches -> the production batches that make up today's stock (for ageing)
results, stock_batches = [], []

for product, p in df.groupby("product"):           # p = all the days for one product
    p = p.set_index("date")                         # use the date as the row label

    # 1. REBUILD THE STOCK LEVEL
    #    The data has no stock column, so we rebuild it: each day stock goes up by what
    #    was produced and down by what was delivered. cumsum() adds this up day by day.
    running_balance = (p["produced_units"] - p["delivered_units"]).cumsum()
    #    The opening stock is unknown, so we use the smallest amount that never lets the
    #    balance go below zero (stock can't be negative).
    stock = running_balance - min(0, running_balance.min())
    avg_stock = stock.tail(28).mean()               # average stock over the last 4 weeks

    # 2. DAILY DEMAND
    #    Use the last 48 working days (~8 weeks). Fridays are the weekly holiday in
    #    Bangladesh (day_of_week 4), so they are left out or they would drag the average down.
    recent = p[p.index.day_of_week != 4]["ordered_units"].tail(48)
    daily_avg = recent.mean()                       # average units ordered per working day
    daily_sd = recent.std()                         # how much daily demand jumps around

    # 3. LEAD TIME
    #    We have no supplier lead times, so we use the average gap (in days) between
    #    production runs: that is how long you wait for the next batch.
    #    clip(1, 30) keeps it between 1 and 30 days; with fewer than 2 runs we assume 30.
    run_dates = p.index[p["produced_units"] > 0]
    if len(run_dates) > 1:
        lead_time = float(np.clip(pd.Series(run_dates).diff().dt.days.mean(), 1, 30))
    else:
        lead_time = 30.0

    # 4. SAFETY STOCK AND REORDER POINT
    #    Safety stock = z x daily variation x square root of lead time
    #      (the buffer that covers demand being higher than average while you wait)
    #    Reorder point = average demand during the lead time + safety stock
    #      (when stock falls to this level, trigger the next production run)
    safety_stock = Z * daily_sd * np.sqrt(lead_time)
    reorder_point = daily_avg * lead_time + safety_stock

    # 5. STOCK AGEING (FIFO = first in, first out)
    #    With FIFO the oldest units are shipped first, so the stock left today comes from
    #    the most recent production runs. Walk back from the newest run until today's
    #    stock is covered, and record how old each batch is.
    to_cover = stock.iloc[-1]                       # stock on the last day
    newest_first = p.loc[p["produced_units"] > 0, "produced_units"][::-1]   # [::-1] = reverse order
    for run_date, qty in newest_first.items():
        if to_cover <= 0:
            break                                   # all of today's stock is accounted for
        units = min(qty, to_cover)                  # take the whole batch, or just what is left
        stock_batches.append({"product": product, "units": units,
                              "age_days": (last_day - run_date).days})
        to_cover -= units

    # 6. SAVE ONE ROW OF RESULTS FOR THIS PRODUCT
    results.append({
        "product": product,
        "product_group": p["product_group"].iloc[0],
        "avg_stock_28d": round(avg_stock),
        "avg_daily_demand": round(daily_avg, 2),
        "sd_daily_demand": round(daily_sd, 2),
        "lead_time_days": round(lead_time, 2),
        "safety_stock": round(safety_stock),
        "reorder_point": round(reorder_point),
        # days of cover = how many days the average stock would last at average demand
        "days_of_cover": round(avg_stock / daily_avg, 1) if daily_avg else None,
        # fill rate = share of ordered units that were actually delivered
        "fill_rate_pct": round(p["delivered_units"].sum() / p["ordered_units"].sum() * 100, 1),
        # production vs orders: under 100% = not making enough, well over 100% = overproducing
        "production_vs_orders_pct": round(p["produced_units"].sum() / p["ordered_units"].sum() * 100, 1),
        "ordered_tonnes": p["ordered_tonnes"].sum(),   # used for ABC (weight, as there are no prices)
    })

inv = pd.DataFrame(results)                         # turn the list of rows into a table
print(f"Calculated {len(inv)} products")

In [7]:
# --- Cell 3: risk flags, ABC/XYZ segments and stock age bands --------------------------------
# Risk flag (first match wins): poor service, then low stock, then too much stock
inv["risk"] = np.select(
    [inv["fill_rate_pct"] < 95,
     inv["avg_stock_28d"] < inv["safety_stock"],
     (inv["days_of_cover"] > 14) & (inv["days_of_cover"] > 3 * inv["lead_time_days"])],
    ["Service risk (fill rate < 95%)", "Low cover (below safety stock)", "Excess risk"],
    default="Healthy")

# ABC: A = products making up the first 80% of tonnes ordered, B = next 15%, C = last 5%
inv = inv.sort_values("ordered_tonnes", ascending=False)
cumulative_share = inv["ordered_tonnes"].cumsum() / inv["ordered_tonnes"].sum()
inv["abc"] = np.select([cumulative_share <= 0.80, cumulative_share <= 0.95], ["A", "B"], "C")

# XYZ: how steady weekly demand is (coefficient of variation = std dev / average)
weekly = df.groupby(["product", df["date"].dt.to_period("W-WED")])["ordered_units"].sum()
cv = weekly.groupby("product").std() / weekly.groupby("product").mean()
inv["xyz"] = np.select([inv["product"].map(cv) <= 0.5, inv["product"].map(cv) <= 1.0], ["X", "Y"], "Z")
inv["segment"] = inv["abc"] + inv["xyz"]

# Stock age bands
batches = pd.DataFrame(stock_batches)
batches["age_band"] = pd.cut(batches["age_days"], [-1, 30, 90, 180, 10_000], labels=["0-30", "31-90", "91-180", "180+"])

inv.to_csv("outputs/inventory_health.csv", index=False)
batches.to_csv("outputs/stock_ageing_batches.csv", index=False)

print(inv["risk"].value_counts(), "\n")
print("Segments:", inv["segment"].value_counts().sort_index().to_dict())
print("Units by age band:", batches.groupby("age_band", observed=False)["units"].sum().round().to_dict())
inv[inv["risk"] != "Healthy"][["product", "segment", "risk", "fill_rate_pct", "production_vs_orders_pct",
                               "avg_stock_28d", "safety_stock", "days_of_cover", "lead_time_days"]]

risk
Healthy                           23
Low cover (below safety stock)     4
Service risk (fill rate < 95%)     3
Excess risk                        1
Name: count, dtype: int64 

Segments: {'AX': 6, 'AY': 1, 'BX': 7, 'BY': 1, 'CX': 5, 'CY': 9, 'CZ': 2}
Units by age band: {'0-30': 140501.0, '31-90': 813.0, '91-180': 0.0, '180+': 0.0}


,product,segment,risk,fill_rate_pct,production_vs_orders_pct,avg_stock_28d,safety_stock,days_of_cover,lead_time_days
20,POV500M24P,BX,Low cover (below safety stock),99.9,98.5,1775,1934,1.6,1.46
19,POV005L04P,CY,Low cover (below safety stock),99.1,98.7,571,598,1.9,2.26
9,MAR02K12P,CX,Service risk (fill rate < 95%),88.0,85.2,545,417,4.3,2.31
10,MASR025K,CY,Service risk (fill rate < 95%),85.8,83.5,392,463,2.6,1.91
6,MAHS025K,CY,Excess risk,96.9,151.5,2670,173,63.8,2.91
4,ATWWP002K12P,CY,Service risk (fill rate < 95%),71.0,71.3,154,415,6.8,30.00
5,MAC1K25P,CZ,Low cover (below safety stock),98.2,105.0,18,71,3.9,12.71
7,MAP1K25P,CZ,Low cover (below safety stock),98.0,102.5,14,572,1.4,30.00


In [10]:
# --- Cell 4: SAP-ready proposal: safety stock (MARC-EISBE) and reorder point (MARC-MINBE) -----
# In SAP, safety stock and reorder point live in table MARC (plant data for each material):
#   EISBE = safety stock, MINBE = reorder point. Here we write our numbers in that layout,
#   so they could be uploaded to SAP after a planner has reviewed them.
marc = pd.read_parquet("data/processed/marc.parquet")              # product (MATNR) -> plant (WERKS)
sap = marc.merge(inv, left_on="MATNR", right_on="product")         # join plant and results by product
sap = sap.rename(columns={"safety_stock": "EISBE", "reorder_point": "MINBE"})
sap = sap[["MATNR", "WERKS", "EISBE", "MINBE", "segment", "risk"]].sort_values("MATNR")
sap.to_csv("outputs/sap_marc_safety_stock_proposal.csv", index=False)
sap.head(10)

,MATNR,WERKS,EISBE,MINBE,segment,risk
15,AT5X5K,1921,1238,2202,AY,Healthy
7,ATN01K24P,1920,4913,9326,AX,Healthy
8,ATN02K12P,1911,1166,2127,BX,Healthy
19,ATWWP001K24P,1920,460,1718,CY,Healthy
27,ATWWP002K12P,1920,415,1101,CY,Service risk (fill rate < 95%)
29,MAC1K25P,1915,71,129,CZ,Low cover (below safety stock)
24,MAHS025K,1919,173,295,CY,Excess risk
28,MAP1K25P,1911,572,869,CZ,Low cover (below safety stock)
10,MAR01K24P,1911,1125,2109,BX,Healthy
9,MAR02K12P,1911,417,707,CX,Service risk (fill rate < 95%)
